AI for the Arts and Humanities (B)

Student ID: 2911567

_2.HowWeTellStories_AI_Human_Demonstration.ipynb_

# **How We Tell Stories:  Exploring AI and Human Interpretation of a 19th Century Continental Tour through Textual and Visual Reconstructions**

Below is sample code that could be used to create an exhibit for Andrew Macgeorge’s Journal of a continental tour, 1867-1868. It demonstrates how his works could be reproduced through a combination of AI and human collaboration, while also exploring the value of these methods and how they might influence the meaning and intent of his original work.

_Please note: This code is demonstrative for a proof of concept rather than finalised code suitable for a public exhibition._

## Initial Setup

### Install the Necessary Libraries

This section installs the required Python libraries, including `torch` for numerical computation, `transformers` for pre-trained models (like BLIP for image captioning), and `diffusers` for image generation (like Stable Diffusion).

In [ ]:
!pip install torch transformers diffusers

### Import the Libraries and Models

This imports specific classes and functions from the installed libraries, such as `BlipProcessor`, `BlipForConditionalGeneration` for image captioning, `StableDiffusionPipeline` for image generation, `PIL` for image handling, `torch` for computational frameworks and `google.colab` for handling files.

In [ ]:
from transformers import BlipProcessor, BlipForConditionalGeneration
from diffusers import StableDiffusionPipeline
from PIL import Image
import torch
from google.colab import files

## STAGE 1: AI and human captioning and reconstruction of the same image

AI and the user are provided witht the same image from the Journal of a Continental Tour (1867–1868) and generate their own textual descriptions (caption).

The captions are then used as a prompt for image generation, producing two reconstructed versions that can be compared to the original drawing.

This stage examines how both humans and machines ‘see’ the same image, translate this information into text, and use this to produce re-creations. This encourages the user to consider how each reshapes the original intent and meaning of the author and the image.

_Each step is broken down by the **coding concepts**, which provides technical explanation and how it fits into ML workflows._
<br>
_Below the code is the **exhibition context** of how it would function and how a user would engage with it._



### Step 1: Upload and display image


This cell allows you to upload an image from your local machine (purely for demonstrating how the code works), opens it using Pillow (`PIL`), converts it to RGB format, and then displays the uploaded image within the notebook.

In [ ]:
# Prompts the user to select a file from the repository.
print("Please upload one of the images provided in the repository titled STAGE-1_Step-1_sketch-of-moustache-man.png.")

# Uses `google.colab.files` to upload an image from the local machine.
uploaded = files.upload()  # choose one .jpg or .png
image_path = list(uploaded.keys())[0]

# Uses `PIL.Image` to open the uploaded image and convert it to RGB format.
raw_image = Image.open(image_path).convert('RGB')
# Displays the image using Colab's display function.
display(raw_image)

#### Exhibition Context


Instead of being able to upload any image, the user would choose from a selection of pictures in the book.

### Step 2: Insert human-produced caption

This cell defines the human-written caption that the user would input based on the image they uploaded/selected. This will later be used as a prompt for the Stable Diffusion model (image generation).

In [ ]:
human_caption = "pen drawing of a man with a very long and thin mustache wearing a top hat"


#### Exhibition Context


This is where the user would be asked to simply caption and describe the image they selected in the step before.

### Step 3: Generate an AI caption from the uploaded image

This section loads the BLIP (Bootstrapping Language-Image Pre-training) model for image captioning. It then uses the model to generate a descriptive caption for the uploaded image (`raw_image`).

In [ ]:
# Load BLIP captioning model using `transformers` library.
processor = BlipProcessor.from_pretrained("Salesforce/blip-image-captioning-large")
model = BlipForConditionalGeneration.from_pretrained("Salesforce/blip-image-captioning-large")

In [ ]:
# Uses the BLIP `processor` and `model` (from `transformers`) to generate a caption.
inputs = processor(raw_image, text="", return_tensors="pt")
out = model.generate(**inputs, max_new_tokens=120)
ai_caption = processor.decode(out[0], skip_special_tokens=True)

# Prints AI generated caption for user to see output.
print("AI caption:\n", ai_caption)

#### Exhibition Context


In real time, the user sees the AI generate a caption for the same image they previously captioned themselves in the step before.

### Step 4: Generate Images using AI and human generated captions

This section uses the Stable Diffusion model with both human-generated and AI-generated captions to generate new images. The generated images are then displayed.

In [ ]:
# Load the Stable Diffusion model using `diffusers` library.
# Moves the model to 'cuda' (GPU) if available, otherwise 'cpu' using `torch`.
pipe = StableDiffusionPipeline.from_pretrained("runwayml/stable-diffusion-v1-5")
pipe = pipe.to("cuda" if torch.cuda.is_available() else "cpu")

human_caption_generated_image = None
# Generates images using the Stable Diffusion `pipe` based on AI and human captions.
for label, text in [("AI version", ai_caption), ("Human version", human_caption)]:
    print(f"\nGenerating: {label}")
    image = pipe(text, num_inference_steps=20).images[0]
    if label == "Human version":
        human_caption_generated_image = image
    # Displays the generated image.
    display(image)

#### Exhibition Context


These can then be compared with the original image to show how the ‘re-telling’ (captioning and image creation) differs from the author’s initial work, and to examine how human involvement in the process can influence the final results.

Options for presentation:
- Ask the user to choose which was based on their (human) caption and which was based on the AI. <br>
This would encourage them to reflect on the differences between the two and on what ‘creates’ new content within AI generation, especially as they are based on the same source.
-	Ask how the meaning of the original image has changed. Has this added to the story, distorted it in any way, or removed elements from the original work?


### Step 5: Generate an AI caption based on the human-prompted AI-generated image

Finally, this cell takes the image generated from the human-provided caption and feeds it back into the BLIP image captioning model to see what caption the AI generates for its own creation based on a human's prompt.

In [ ]:
# Generate AI caption for the human-generated image
print("\nGenerating AI caption for the human-generated image...")
# Uses the BLIP `processor` and `model` (from `transformers`) to generate a caption for the human-generated image.
inputs_human_gen = processor(human_caption_generated_image, text="", return_tensors="pt")
out_human_gen = model.generate(**inputs_human_gen, max_new_tokens=120)
ai_caption_human_gen = processor.decode(out_human_gen[0], skip_special_tokens=True)

print("AI caption for human-generated image:\n", ai_caption_human_gen)

#### Exhibition Context


This final step is to deepen the scrutiny of AI's role in this process. By getting the AI to caption the newly generated image, it examines how it understands what it produced in response to the human prompt.

<br>

By getting the AI to review its own generation, it provides us insight into the influence the human input had in the process. Has what they wrote been reflected?

## STAGE 2: AI and human image generation from the same description

AI and the user are provided with the same textual description and both generate their own image. This can be directly compared with the description they were provided. This is designed to highlight the differences and similarities in how human and AI interpret and then re-create a textual description into an image.

This stage asks the user to engage creatively by drawing. This is not a test of ability, but rather a demonstration that what humans produce is based on what they know and their skills, much like a trained AI model.

The uniqueness and background of each approach can be reflected in how it alters or changes the original work.


_Each step is broken down by the **coding concepts**, which provides technical explanation and how it fits into ML workflows._
<br>
_Below the code is the **exhibition context** of how it would function and how a user would engage with it._

### Step 1: Provide the description
This cell defines the predetermined description that is based off an image in the book (The image is being kept hidden from user at this stage so all they have to go off is this text)

In [ ]:
# Defines in the code the textual description for the image generation and captioning tasks.
original_description = """
Three seasick passengers. Two are leaning over the ship's railing, and the third has a bowl.
""".strip()

print("Predetermined description:\n") # Prints a header for the original description.
print(original_description) # Displays the original description to the user.

#### Exhibition Context

This is a description the user will be asked to use to draw an image. In this example, it is a rather abract image/description to almost encourage diversion from the source material. This should highlight how the creative processes of humans and AI can produce completely different content based on the same source material.

The purpose of keeping the image hidden is to prevent bias in their drawing, encouraging them to create their interpretation of the description.

### Step 2: Upload human-produced image

This cell allows the user to upload an image of their drawing based on the above description from their local machine, opens it using Pillow (PIL), converts it to RGB format, and then displays the uploaded image within the notebook.



In [ ]:
# Prompts the user to upload their drawing.
print("Please upload the drawing YOU created from the description.")
# Find STAGE-2_Step-2_human-sketch-example.png in the repository for an example.

# Uses `google.colab.files` to upload an image from the local machine.
uploaded = files.upload() # either .png or .jpg
human_image_path = list(uploaded.keys())[0]

# Uses `PIL.Image` to open the uploaded image and convert it to RGB format.
human_image = Image.open(human_image_path).convert('RGB')

print("\nYour (human) drawing:") # Prints a header for the human-drawn image.
display(human_image) # Displays the human-drawn image within the notebook.

#### Exhibition Context

The user would be prompted to upload a drawing they created based on the hidden description. This will allow for a direct comparison between human and AI interpretations of the same text.

The user's drawing ability does not matter here. Whether it closely resembles the textual description or not will highlight both the way they interpret the text and how their personal experience and skills affect their ability to create content.

Much like an AI is trained, humans can be considered trained as well.

### Step 3: Generate an AI image from the description

This section uses the Stable Diffusion model (with the same caption provided to the user before) to generate an image. The generated image is then displayed.

In [ ]:
# Load the Stable Diffusion model using `diffusers` library.
# Moves the model to 'cuda' (GPU) if available, otherwise 'cpu' using `torch`.
pipe = StableDiffusionPipeline.from_pretrained("runwayml/stable-diffusion-v1-5")
pipe = pipe.to("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
# Informs user that AI image generation is starting.
print("\nGenerating AI image from the description...")
# Uses the Stable Diffusion `pipe` to generate an image from the `original_description`.
ai_generated_image = pipe(original_description, num_inference_steps=25).images[0]
# Displays AI-generated image.
display(ai_generated_image)

#### Exhibition Context

Here, the AI generates its own image based on the exact same description that the user (human) used to create their drawing. This provides a clear point of comparison for how each 'sees' and translates text into visuals.

The quality of the output is not so important here, rather the user should focus on how it diverges from their own drawing.

What informed the AI to produce an image to look like this? What informed the user to draw their image? They both bring their own knowledge, experience, skills that informs their creative process and final design.

### Step 4: Upload the original image for comparison

In [ ]:
# Prompts the user to upload the original drawing, available in the repository as STAGE-2_Step-4_seasick-passengers.png.
print("Now upload the original drawing (the real one - STAGE-2_Step-4_seasick-passengers.png).")

# Uses `google.colab.files` to open file uploads.
uploaded2 = files.upload()
original_image_path = list(uploaded2.keys())[0]

 # Opens the original image using PIL and converts it to RGB format.
original_image = Image.open(original_image_path).convert('RGB')

print("\nOriginal drawing:") # Prints a header for the original drawing.
display(original_image) # Displays the original drawing.

#### Exhibition Context

After both the human and AI have produced their images, the original drawing from the book is revealed.

This allows for a three-way comparison: the **human's** interpretation, the **AI's** interpretation, both based on the same description of the **original** artist's work.

Here it is brought back to the purpose of the exhibit, which is to ask how these recreations have changed the meaning of the original images in the journal. When we retell a story or produce something based on a description, the human or AI that makes it adds their own understanding to it, altering and transforming it from the original.

### Step 5: Generate an AI description of the human-produced image

This section loads the BLIP (Bootstrapping Language-Image Pre-training) model for image captioning. It uses the model to generate a descriptive caption of the human-drawn image.

In [ ]:
processor = BlipProcessor.from_pretrained("Salesforce/blip-image-captioning-large")
model = BlipForConditionalGeneration.from_pretrained("Salesforce/blip-image-captioning-large")

In [ ]:
# Uses the BLIP `processor` and `model` (from `transformers`) to generate a caption for the human-generated image.
inputs = processor(human_image, return_tensors="pt")
caption_ids = model.generate(**inputs, max_new_tokens=120)
ai_description_of_human = processor.decode(caption_ids[0], skip_special_tokens=True)

print("\nAI description of the HUMAN drawing:") # Prints a header for the AI's description of the human drawing.
print(ai_description_of_human) # Displays the AI-generated description of the human's drawing.

#### Exhibition Context

This final step involves the AI generating a caption for the image created by the human. This provides additional insight into how the AI 'understands' and describes human artistry, and how its interpretation aligns or diverges from the original textual description.

This is particularly interesting as this is 2 steps removed from the original work by human interpretation. The first being the textual description of the image (that was provided to the user), and the second is the recreation of the image through the user's drawing.